# How similar are the old official drone photos and the new high-altitude ones?
Question: could a method that learns or tunes from the old photos (96-347 m, Mumbai area) be tested on the new set (100-2017 m, Rajasthan), or are they too different? Everything is measured on the 500 + 500 photos, nothing is run through the pipeline.
1. Metadata: camera, tilt, heading, height, place.
2. Image statistics per photo (grey range, contrast, sharpness, noise, edge density, lens vignetting, spectrum), old photos against new photos in height groups. The new photos at 100-350 m are the fair comparison for the old ones (same height, different place), the higher groups show what the height itself does.
3. A learned view: the pipeline's own DINOv2 ViT-S/14 embeds every photo; a classifier tries to tell old from new (AUC 0.5 = indistinguishable, 1.0 = trivially different), the Frechet distance and the nearest-neighbour mix measure how far apart the two clouds are.
Reading rule written before the results: "similar enough to test a method on" needs (a) the matched-height new photos (100-350 m) not separable from the old ones by a simple classifier (AUC under about 0.8) AND (b) the image statistics inside the old photos' range; if not, the new set tests robustness to a different scene, not the same kind of data.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
HEADLESS = ROOT.parent / "Main_Pipeline_Headless"
sys.path.insert(0, str(HEADLESS))
from geoloc import photo, flat_field
OLD = Path(os.environ.get("OLD_DIR", str(ROOT.parent / "Code_Testing/dataset/d1/images")))
NEW = Path(os.environ.get("NEW_DIR", "/tmp/claude-1000/sim_new/images"))
NEW_META = Path(os.environ.get("NEW_META", str(NEW.parent / "metadata.csv")))
OUT = ROOT / "tests/Test_Iteration_12/results/similarity"; OUT.mkdir(parents=True, exist_ok=True)
old_meta = pd.read_csv(OLD / "metadata.csv"); new_meta = pd.read_csv(NEW_META)
old_meta["set"] = "old official"; new_meta["set"] = "new high-altitude"
for t, d in ((old_meta, OLD), (new_meta, NEW)):
    t["path"] = [str(d / f) for f in t.filename]; t["height"] = t.alt
GROUPS = [("old official (96-347 m)", None), ("new 100-350 m", (100, 350)), ("new 350-800 m", (350, 800)), ("new 800-1300 m", (800, 1300)), ("new 1300-2100 m", (1300, 2100))]
def group_of(row):
    if row.set == "old official": return GROUPS[0][0]
    for name, rng in GROUPS[1:]:
        if rng[0] <= row.height < rng[1]: return name
all_meta = pd.concat([old_meta, new_meta], ignore_index=True); all_meta["group"] = all_meta.apply(group_of, axis=1)
print(all_meta.groupby("group").size().to_string())

## 1. Metadata

In [ ]:
def tilt(t): return np.hypot(((t.pitch + 180) % 360) - 180, ((t.roll + 180) % 360) - 180)
rows = []
for name, t in (("old official", old_meta), ("new high-altitude", new_meta)):
    rows.append({"set": name, "photos": len(t), "size (px)": "320 x 256", "cam_fx": round(t.cam_fx.median(), 1), "cam_cx, cam_cy": "%.1f, %.1f" % (t.cam_cx.median(), t.cam_cy.median()), "k1": round(t.cam_k1.median(), 3),
                 "height min / median / max (m)": "%.0f / %.0f / %.0f" % (t.height.min(), t.height.median(), t.height.max()),
                 "tilt median / p95 / max (deg)": "%.1f / %.1f / %.1f" % (tilt(t).median(), tilt(t).quantile(.95), tilt(t).max()),
                 "latitude range": "%.3f - %.3f" % (t.lat_gtruth.min(), t.lat_gtruth.max()), "longitude range": "%.3f - %.3f" % (t.lon_gtruth.min(), t.lon_gtruth.max()),
                 "ground size of a photo pixel (m)": "%.2f - %.2f" % (t.height.min() / t.cam_fx.median(), t.height.max() / t.cam_fx.median())})
display(pd.DataFrame(rows).set_index("set").T)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].hist([old_meta.height, new_meta.height], bins=np.arange(0, 2200, 100), label=["old", "new"]); axes[0].set_xlabel("flight height (m)"); axes[0].legend()
axes[1].hist([tilt(old_meta), tilt(new_meta)], bins=np.arange(0, 24, 1.5), label=["old", "new"]); axes[1].set_xlabel("tilt (deg)")
axes[2].hist([old_meta.yaw % 360, new_meta.yaw % 360], bins=np.arange(0, 361, 30), label=["old", "new"]); axes[2].set_xlabel("yaw (deg)")
fig.tight_layout(); fig.savefig(OUT / "fig1_metadata.png", dpi=120); plt.show(); plt.close(fig)

## 2. Image statistics per photo

In [ ]:
def radial_psd(frame, bins=24):
    f = frame.astype(np.float64); f = f - f.mean(); f *= np.outer(np.hanning(f.shape[0]), np.hanning(f.shape[1]))
    power = np.abs(np.fft.fftshift(np.fft.fft2(f))) ** 2
    rows, cols = np.mgrid[0:f.shape[0], 0:f.shape[1]]; radius = np.hypot((rows - f.shape[0] / 2) / f.shape[0], (cols - f.shape[1] / 2) / f.shape[1])
    edges = np.linspace(0.02, 0.5, bins + 1); out = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (radius >= lo) & (radius < hi); out.append(power[m].mean() if m.any() else np.nan)
    return np.array(out) / (np.nansum(out) + 1e-12), (edges[:-1] + edges[1:]) / 2

def photo_stats(path, cx, cy):
    raw = photo.load_photo(path).astype(np.float64)
    lo, hi = np.percentile(raw, [1, 99])
    stretched = np.clip((raw - lo) / (hi - lo + 1e-9), 0, 1).astype(np.float32)
    surface = flat_field.radial_surface(raw, cx, cy, 3)
    explained = 1.0 - np.var(raw - surface) / (np.var(raw) + 1e-12)
    flat = raw - (surface - surface.mean()); flo, fhi = np.percentile(flat, [1, 99]); flat_s = np.clip((flat - flo) / (fhi - flo + 1e-9), 0, 1).astype(np.float32)
    def edge_share(img):
        g = cv2.GaussianBlur(img, (0, 0), 1.0); m = np.hypot(cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1)); return float((m > 0.2).mean())
    hist = np.histogram(stretched, bins=64, range=(0, 1))[0] / stretched.size; entropy = float(-(hist[hist > 0] * np.log2(hist[hist > 0])).sum())
    lap = cv2.Laplacian(flat_s, cv2.CV_32F); mad_noise = float(1.4826 * np.median(np.abs(lap - np.median(lap))))
    psd, freq = radial_psd(flat_s)
    return dict(raw_range=float(hi - lo), grey_levels=int(len(np.unique(raw))), std_flat=float(flat_s.std()), entropy_bits=entropy, sharpness=float(lap.var()), noise_mad=mad_noise,
                edge_share_raw=edge_share(stretched), edge_share_flat=edge_share(flat_s), vignette_explained=float(explained), hf_power=float(psd[freq > 0.25].sum()), psd=psd)

records = []
for i, r in all_meta.iterrows():
    s = photo_stats(r.path, r.cam_cx, r.cam_cy); s.update(group=r.group, set=r.set, height=r.height, filename=r.filename); records.append(s)
S = pd.DataFrame(records); PSD = np.stack(S.psd.values); S = S.drop(columns="psd")
S.to_csv(OUT / "per_photo_stats.csv", index=False)
order = [g[0] for g in GROUPS]
cols = ["raw_range", "std_flat", "entropy_bits", "sharpness", "noise_mad", "edge_share_raw", "edge_share_flat", "vignette_explained", "hf_power"]
med = S.groupby("group")[cols].median().reindex(order); display(med.round(4))
med.to_csv(OUT / "stats_median_by_group.csv")

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
for ax, col in zip(axes.ravel(), cols):
    ax.boxplot([S[S.group == g][col].values for g in order], labels=["old", "new\n100-350", "new\n350-800", "new\n800-1300", "new\n1300-2100"], showfliers=False)
    ax.set_title(col); ax.grid(alpha=.3)
fig.suptitle("image statistics per photo: old official against the new set in height groups"); fig.tight_layout(); fig.savefig(OUT / "fig2_stats_boxplots.png", dpi=120); plt.show(); plt.close(fig)
freq = radial_psd(np.zeros((256, 320)) + np.random.default_rng(0).random((256, 320)))[1]
fig, ax = plt.subplots(figsize=(8, 5))
for g in order:
    ax.loglog(freq, np.nanmedian(PSD[(S.group == g).values], axis=0), label=g)
ax.set_xlabel("spatial frequency (cycles per pixel)"); ax.set_ylabel("share of power"); ax.legend(); ax.grid(alpha=.3, which="both"); ax.set_title("radially averaged power spectrum (vignette removed, median per group)")
fig.tight_layout(); fig.savefig(OUT / "fig3_spectrum.png", dpi=120); plt.show(); plt.close(fig)

In [ ]:
rng = np.random.default_rng(1); tiles = []
for g in order:
    sel = all_meta[all_meta.group == g].sample(5, random_state=3)
    row = []
    for _, r in sel.iterrows():
        frame = photo.normalise_frame(photo.load_photo(r.path)); row.append(cv2.resize(frame, (240, 192)))
    tiles.append(np.concatenate(row, axis=1))
pic = (np.concatenate(tiles, axis=0) * 255).astype(np.uint8)
cv2.imwrite(str(OUT / "fig4_sample_photos.png"), pic)
fig, ax = plt.subplots(figsize=(15, 9.5)); ax.imshow(pic, cmap="gray"); ax.set_yticks([96 + 192 * i for i in range(len(order))]); ax.set_yticklabels(order); ax.set_xticks([]); ax.set_title("5 random photos per group as the pipeline sees them (percentile stretch + CLAHE)")
fig.tight_layout(); fig.savefig(OUT / "fig4_sample_photos_labelled.png", dpi=100); plt.show(); plt.close(fig)

## 3. A learned view: DINOv2 embeddings of every photo

In [ ]:
import timm
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
encoder = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True, pretrained_cfg_overlay=dict(file=str(HEADLESS / "models/dinov2_vits14_timm.safetensors")), img_size=224).to(DEVICE).eval()
mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1); std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
feats = []
with torch.no_grad():
    for start in range(0, len(all_meta), 32):
        batch = [cv2.resize(photo.normalise_frame(photo.load_photo(p)), (224, 224)) for p in all_meta.path[start:start + 32]]
        x = torch.from_numpy(np.stack(batch)).float().to(DEVICE).unsqueeze(1).repeat(1, 3, 1, 1); x = (x - mean) / std
        tokens = encoder.forward_features(x); feats.append(torch.cat([tokens[:, 0], tokens[:, 1:].mean(1)], dim=1).float().cpu().numpy())
F = np.concatenate(feats); np.save(OUT / "dino_features.npy", F); print(F.shape)

In [ ]:
is_new = (all_meta.set == "new high-altitude").values; group = all_meta.group.values
def auc_of(mask_a, mask_b, seed=0):
    idx = np.where(mask_a | mask_b)[0]; y = mask_b[idx].astype(int)
    model = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=2000))
    p = cross_val_predict(model, F[idx], y, cv=StratifiedKFold(5, shuffle=True, random_state=seed), method="predict_proba")[:, 1]
    return roc_auc_score(y, p)
def frechet(a, b):
    from scipy.linalg import sqrtm
    ca = np.cov(a, rowvar=False) + 1e-3 * np.eye(a.shape[1]); cb = np.cov(b, rowvar=False) + 1e-3 * np.eye(a.shape[1])
    root = sqrtm(ca @ cb); root = root.real if np.iscomplexobj(root) else root
    return float(((a.mean(0) - b.mean(0)) ** 2).sum() + np.trace(ca + cb - 2 * root))
old_mask = group == order[0]; rows = []
for g in order[1:]:
    m = group == g
    nn = []
    from sklearn.neighbors import NearestNeighbors
    Z = StandardScaler().fit_transform(F); knn = NearestNeighbors(n_neighbors=6).fit(Z); _, idx = knn.kneighbors(Z[m]); nn_old = float(np.mean(old_mask[idx[:, 1:]]))
    rows.append({"new group": g, "photos": int(m.sum()), "AUC old vs this group": round(auc_of(old_mask, m), 3), "Frechet distance to old": round(frechet(F[old_mask], F[m]), 1),
                 "share of 5 nearest neighbours that are old photos": round(nn_old, 3), "(chance level, if mixed)": round(old_mask.sum() / (old_mask.sum() + m.sum()), 3)})
rows.append({"new group": "ALL new", "photos": int(is_new.sum()), "AUC old vs this group": round(auc_of(old_mask, is_new), 3), "Frechet distance to old": round(frechet(F[old_mask], F[is_new]), 1), "share of 5 nearest neighbours that are old photos": np.nan, "(chance level, if mixed)": np.nan})
# reference: two random halves of the OLD photos (how far apart are two samples of the same data)
half = np.random.default_rng(0).permutation(np.where(old_mask)[0]); a, b = half[:len(half) // 2], half[len(half) // 2:]
ma = np.zeros(len(F), bool); mb = np.zeros(len(F), bool); ma[a] = True; mb[b] = True
rows.append({"new group": "REFERENCE: old half A vs old half B", "photos": len(b), "AUC old vs this group": round(auc_of(ma, mb), 3), "Frechet distance to old": round(frechet(F[a], F[b]), 1), "share of 5 nearest neighbours that are old photos": np.nan, "(chance level, if mixed)": np.nan})
R3 = pd.DataFrame(rows); display(R3); R3.to_csv(OUT / "dino_distances.csv", index=False)
pca = PCA(n_components=2).fit(F); P = pca.transform(F)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for g, c in zip(order, ["k", "tab:blue", "tab:green", "tab:orange", "tab:red"]):
    axes[0].scatter(P[group == g, 0], P[group == g, 1], s=8, c=c, alpha=.6, label=g)
axes[0].legend(fontsize=8); axes[0].set_title("DINOv2 embeddings of the photos, first two principal components (%.0f %% of variance)" % (100 * pca.explained_variance_ratio_.sum()))
sc = axes[1].scatter(P[:, 0], P[:, 1], s=8, c=np.where(is_new, np.log10(all_meta.height.values), np.nan), cmap="viridis"); axes[1].scatter(P[~is_new, 0], P[~is_new, 1], s=8, c="lightgrey"); plt.colorbar(sc, ax=axes[1], label="log10 height of the new photos (grey = old)")
fig.tight_layout(); fig.savefig(OUT / "fig5_dino_pca.png", dpi=120); plt.show(); plt.close(fig)

## 4. Verdict (numbers computed above)

In [ ]:
a_matched = R3.loc[R3["new group"] == "new 100-350 m", "AUC old vs this group"].iloc[0]; a_all = R3.loc[R3["new group"] == "ALL new", "AUC old vs this group"].iloc[0]; a_ref = R3.iloc[-1]["AUC old vs this group"]
print("classifier AUC, old against new photos at 100-350 m (same height, other place): %.3f" % a_matched)
print("classifier AUC, old against all new photos: %.3f" % a_all)
print("reference, two halves of the old set: %.3f" % a_ref)
print()
print("verdict by the pre-written rule: (a) matched-height AUC under about 0.8? ->", "yes" if a_matched < 0.8 else "NO")
inside = []
for col in ["raw_range", "std_flat", "sharpness", "edge_share_flat", "vignette_explained", "hf_power"]:
    o = S[S.group == order[0]][col]; n = S[S.group == order[1]][col]
    inside.append((col, float(n.median()), float(o.quantile(.05)), float(o.quantile(.95)), bool(o.quantile(.05) <= n.median() <= o.quantile(.95))))
print("(b) median of the new 100-350 m photos inside the 5-95 % range of the old photos:")
for c in inside: print("   %-20s new %.4f | old 5-95 %% %.4f to %.4f | %s" % (c[0], c[1], c[2], c[3], "inside" if c[4] else "OUTSIDE"))